# Notebook 03 — Norms, Broadcasting, and Distances

## Objective

Build L1/L2 norms, cosine similarity, feature scaling, and pairwise Euclidean distances; inspect how they affect nearest-neighbor decisions.

**Prerequisites:** Linear algebra Notebooks 01 and 02: shapes, axes, dot products, and matrix operations.

**Working rule:** Complete the starter cells yourself. Verification cells are deliberately placed after each task and will fail until that task is solved. Keep the assertions; catch only failures that the task explicitly asks you to provoke. Use NumPy plus Python's standard library only. After finishing, restart the kernel and run all cells in order.

**Why this matters:** Distance calculations support nearest neighbors, clustering, and similarity search. A result can be arithmetically correct yet dominated by an arbitrary feature unit.


## Shared synthetic transactions

Columns are transaction amount, recent transaction count, account age in days, and merchant diversity. These data have no fraud labels. Do not describe a nearest neighbor as a fraud prediction.

In [ ]:
import numpy as np

X = np.array([[100., 1., 365., 5.],
              [110., 1., 370., 5.],
              [300., 8., 360., 2.],
              [105., 2., 30., 5.],
              [95., 1., 355., 6.]])
feature_names = ('amount', 'recent_count', 'account_age_days', 'merchant_diversity')
assert X.shape == (5, 4)


## Task 1 — L1 and L2 norms from scratch

### Goal

Measure vector size and connect each norm to a different treatment of large components.

### Required work

1. Implement `l1_norm(vector)` and `l2_norm(vector)` for numerical 1D arrays.
2. Calculate both norms of `[3, -4]` by hand and compare with NumPy.
3. Test a zero vector and reject a matrix passed as a vector.

### Expected output

L1 is `7`, L2 is `5` for `[3, -4]`; both are zero for a zero vector.

### Acceptance criteria

- [ ] Functions validate 1D input.
- [ ] Hand results and NumPy results agree.
- [ ] The explanation contrasts absolute deviations and squared deviations.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def l1_norm(vector):
    # TODO: validate 1D and calculate without np.linalg.norm.
    raise NotImplementedError

def l2_norm(vector):
    # TODO: validate 1D and calculate without np.linalg.norm.
    raise NotImplementedError


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
sample = np.array([3., -4.])
assert np.isclose(l1_norm(sample), 7.)
assert np.isclose(l2_norm(sample), 5.)
assert np.isclose(l1_norm(sample), np.linalg.norm(sample, ord=1))
assert np.isclose(l2_norm(sample), np.linalg.norm(sample, ord=2))
assert l1_norm(np.zeros(2)) == l2_norm(np.zeros(2)) == 0
for fn in (l1_norm, l2_norm):
    try:
        fn(np.ones((1, 2)))
    except ValueError:
        pass
    else:
        raise AssertionError('Expected 1D input only')


## Task 2 — Feature-wise broadcasting and standardization

### Goal

Use axes correctly and make feature scales comparable.

### Required work

1. Compute one mean and population standard deviation per column of `X`.
2. Implement `standardize_columns(data)` that returns the standardized matrix plus means and scales.
3. Define and test a policy for constant columns so no division by zero occurs.
4. Explain why subtracting a row of four means broadcasts across five observations.

### Expected output

A `(5, 4)` standardized matrix whose nonconstant columns have mean approximately 0 and population standard deviation approximately 1.

### Acceptance criteria

- [ ] No input array is mutated.
- [ ] Constant-column behavior is finite and documented.
- [ ] Output shapes are checked.

**Hint:** Hint: A `(4,)` feature summary can broadcast against a `(5, 4)` matrix; a row summary of shape `(5,)` cannot.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def standardize_columns(data):
    # TODO: validate 2D numerical input; return (scaled, means, scales).
    # Decide how a constant feature should be represented.
    raise NotImplementedError


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
Z, means, scales = standardize_columns(X)
assert Z.shape == X.shape and means.shape == scales.shape == (4,)
assert np.all(np.isfinite(Z)) and np.all(np.isfinite(scales))
assert np.allclose(Z.mean(axis=0), 0, atol=1e-12)
assert np.allclose(Z.std(axis=0), 1, atol=1e-12)
constant = np.column_stack((X, np.ones(len(X))))
Z_const, _, _ = standardize_columns(constant)
assert np.all(np.isfinite(Z_const)) and np.allclose(Z_const[:, -1], 0)


## Task 3 — Pairwise Euclidean distances

### Goal

Turn differences between rows into a symmetric distance matrix.

### Required work

1. Implement `pairwise_l2(data)` using explicit row pairs and your L2 norm.
2. Implement a second version with NumPy broadcasting; predict the temporary array shape before running it.
3. Check diagonal, symmetry, and a hand-computed distance for a tiny two-point dataset.

### Expected output

A `(5, 5)` nonnegative distance matrix; distance between `[0, 0]` and `[3, 4]` is `5`.

### Acceptance criteria

- [ ] Loop and broadcast versions agree.
- [ ] Diagonal is zero and matrix is symmetric.
- [ ] No square root of a negative number from roundoff.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def pairwise_l2(data):
    # TODO: validate 2D, loop over pairs, and use l2_norm.
    raise NotImplementedError

def pairwise_l2_broadcast(data):
    # TODO: use row/column singleton axes and NumPy operations.
    raise NotImplementedError


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
tiny = np.array([[0., 0.], [3., 4.]])
assert np.allclose(pairwise_l2(tiny), [[0., 5.], [5., 0.]])
raw_D = pairwise_l2(X)
assert raw_D.shape == (5, 5)
assert np.allclose(raw_D, pairwise_l2_broadcast(X))
assert np.allclose(raw_D, raw_D.T)
assert np.allclose(np.diag(raw_D), 0)
assert np.all(raw_D >= 0)


## Task 4 — Cosine similarity and the zero vector

### Goal

Compare direction rather than magnitude, while defining invalid cases.

### Required work

1. Implement `cosine_similarity(a, b)` for equal-length 1D vectors using a dot product and norms.
2. Choose an explicit zero-vector policy: raise `ValueError` rather than divide by zero.
3. Check identical, orthogonal, and opposite vectors; explain when cosine could be preferable to Euclidean distance.

### Expected output

Similarity values `1`, `0`, and `-1` for the three reference pairs.

### Acceptance criteria

- [ ] Finite results lie in `[-1, 1]` within tolerance.
- [ ] Zero vector and shape mismatch raise `ValueError`.
- [ ] The ML interpretation mentions scale sensitivity.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def cosine_similarity(a, b):
    # TODO: validate 1D equal-length arrays and nonzero norms.
    raise NotImplementedError


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
e1, e2 = np.array([1., 0.]), np.array([0., 1.])
assert np.isclose(cosine_similarity(e1, e1), 1)
assert np.isclose(cosine_similarity(e1, e2), 0)
assert np.isclose(cosine_similarity(e1, -e1), -1)
for a, b in ((e1, np.zeros(2)), (e1, np.ones(3))):
    try:
        cosine_similarity(a, b)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid cosine input was accepted')


## Task 5 — Diagnose a broadcasting and unit mistake

### Goal

See how numerically valid code can express the wrong comparison.

### Required work

1. Show that subtracting `X.mean(axis=1, keepdims=True)` runs but centers rows, not features.
2. Compute raw and standardized distances from transaction 0 to all other transactions.
3. Explain why a column measured in days or money can dominate a raw distance.

### Expected output

Two distance rankings plus a written diagnosis of the incorrect axis.

### Acceptance criteria

- [ ] Wrong row-centering shape is `(5, 4)` but its column means need not be zero.
- [ ] Correct standardization has near-zero feature means.
- [ ] Distance rankings are computed, not guessed.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
# TODO: produce row_centered, standardized_D, raw_ranking, standardized_ranking.
row_centered = standardized_D = raw_ranking = standardized_ranking = None


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
assert row_centered is not None and row_centered.shape == X.shape
assert not np.allclose(row_centered.mean(axis=0), 0)
assert standardized_D is not None and standardized_D.shape == (5, 5)
assert np.allclose(standardized_D, pairwise_l2(Z))
assert raw_ranking is not None and standardized_ranking is not None
assert sorted(raw_ranking) == sorted(standardized_ranking) == [1, 2, 3, 4]
assert np.array_equal(raw_ranking, np.argsort(raw_D[0])[1:])
assert np.array_equal(standardized_ranking, np.argsort(standardized_D[0])[1:])


## Task 6 — Nearest neighbors as an ML preprocessing step

### Goal

Produce reproducible similarity results without claiming to detect fraud.

### Required work

1. Implement `nearest_neighbors(data, query_index, k)` using pairwise distances; exclude the query row.
2. Return neighbor indices ordered by increasing distance, resolving ties by smaller index.
3. Compare raw and standardized top-2 neighbors for transaction 0 and explain the difference.
4. Reject invalid query indices and `k` values.

### Expected output

Two index lists of length 2, with no query index, and a short caution about unlabeled similarity.

### Acceptance criteria

- [ ] Order agrees with a NumPy reference sort.
- [ ] Invalid arguments raise `ValueError`.
- [ ] Explanation does not equate similarity with fraud.

**Deliverable:** Completed starter cell, passing verification cell, and 2–4 sentences below the task explaining the mathematical idea and one ML consequence or limitation. Replace this instruction with your own explanation.


In [ ]:
def nearest_neighbors(data, query_index, k):
    # TODO: validate inputs and return a 1D integer array of indices.
    raise NotImplementedError


### Self-check — run after completing the task

The assertions below are part of the assignment. Do not change expected values to match a wrong implementation.

In [ ]:
for data in (X, Z):
    result = np.asarray(nearest_neighbors(data, 0, 2))
    assert result.shape == (2,) and 0 not in result
    expected = np.lexsort((np.arange(len(data)), pairwise_l2(data)[0]))
    expected = expected[expected != 0][:2]
    assert np.array_equal(result, expected)
for index, k in ((-1, 2), (len(X), 2), (0, 0), (0, len(X))):
    try:
        nearest_neighbors(X, index, k)
    except ValueError:
        pass
    else:
        raise AssertionError('Invalid nearest-neighbor request was accepted')


## Completion check

- [ ] All task cells are implemented and all provided assertions pass.
- [ ] Expected exceptions are caught inside the relevant test cells.
- [ ] **Restart Kernel and Run All** succeeds in notebook order.
- [ ] Each task has a short written explanation in your own words.
- [ ] No real private data, credentials, or large generated files were added.

**Conceptual exit questions**

1. What do L1 and L2 emphasize differently?
2. Which axes does feature-wise standardization operate over?
3. Why can raw Euclidean distance change under a change of units?
4. What does cosine similarity do with magnitude?
5. Why is a nearest neighbor not automatically a fraud prediction?

**Final deliverable:** A notebook with verified norm, scaling, distance, cosine, and neighbor functions plus an interpretation of unit sensitivity.
